In [ ]:
# batched_shots_gpu=False on its own, on the original pinned versions.

!pip install -q qiskit==1.4.6 qiskit-aer-gpu==0.15.1 qiskit-algorithms==0.4.0 qiskit-optimization==0.7.0 qiskit-ibm-runtime==0.29.0 2>&1 | tail -5

import sys, traceback, numpy as np
from qiskit.circuit.library import TwoLocal
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit_aer.noise import NoiseModel
from qiskit_ibm_runtime.fake_provider import FakeCairoV2
from qiskit_algorithms import SamplingVQE
from qiskit_algorithms.optimizers import SPSA
from qiskit.quantum_info import SparsePauliOp
from qiskit import transpile

def p(*a):
    print(*a, flush=True)

import qiskit, qiskit_aer
p("qiskit:", qiskit.__version__, " qiskit_aer:", qiskit_aer.__version__)
from qiskit_aer import AerSimulator
p("available devices:", AerSimulator().available_devices())

n_qubits = 15
rng = np.random.default_rng(0)
terms = []
for i in range(n_qubits):
    lab = ["I"] * n_qubits; lab[i] = "Z"
    terms.append(("".join(lab), float(rng.uniform(-2, 2))))
for i in range(n_qubits - 1):
    lab = ["I"] * n_qubits; lab[i] = "Z"; lab[i+1] = "Z"
    terms.append(("".join(lab), float(rng.uniform(-2, 2))))
ising_op = SparsePauliOp.from_list(terms)

p("Building noise model...")
cairo_nm = NoiseModel.from_backend(FakeCairoV2())
p("noise model error channels:", len(cairo_nm.to_dict()["errors"]))

p("\nSPSA + GPU + batched_shots_gpu=False")
sampler = AerSampler(
    seed=42,
    options={"backend_options": {
        "noise_model": cairo_nm, "method": "statevector", "device": "GPU",
        "batched_shots_gpu": False,
        "max_parallel_threads": 0, "max_parallel_experiments": 0,
    }},
)
sampler.options.default_shots = 2000
ansatz = TwoLocal(num_qubits=n_qubits, rotation_blocks=["ry","rz"], entanglement_blocks="cz",
                   entanglement="circular", reps=3)
p("Transpiling ansatz...")
ansatz_d = transpile(ansatz.decompose(reps=10), backend=sampler._backend, optimization_level=1)
init = np.random.default_rng(1).uniform(-np.pi, np.pi, size=ansatz_d.num_parameters)

def cb(count, params, value, meta):
    p(f"  eval {count}: energy={value:.4f}")

p("Starting SPSA optimization (maxiter=15)...")
try:
    vqe = SamplingVQE(sampler=sampler, ansatz=ansatz_d, optimizer=SPSA(maxiter=15), initial_point=init, callback=cb)
    result = vqe.compute_minimum_eigenvalue(ising_op)
    p(f"\nworked: eigenvalue={float(np.real(result.eigenvalue)):.4f}")
except Exception:
    p("\nfailed, full traceback:")
    traceback.print_exc(file=sys.stdout)
    sys.stdout.flush()